# 01_configuracion

In [ ]:
!pip install ezdxf matplotlib numpy

In [ ]:
# =====================================================================
# CELDA 2: CONFIGURACIÓN E IMPORTS
# =====================================================================

# Imports principales
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import ezdxf
import math
import csv
import pickle
import os
from ezdxf.math import Vec2
from collections import defaultdict
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
from google.colab import drive

# Configuración global del notebook
CONFIG = {
    # Tolerancias para cálculos geométricos
    'tolerances': {
        'geometric': 0.01,
        'slope': 0.0001,
        'position': 0.01,
        'vertical': 0.01,
        'horizontal': 0.001,
        'y_coordinate': 0.001
    },

    # Rutas de archivos (configurables)
    'paths': {
        'data_folder': '/content/tower_data',
        'output_folder': '/content/output',
        'drive_folder': '/content/drive/MyDrive/MiCarpetaColab',
        'temp_folder': '/content/temp'
    },

    # Configuración de visualización
    'plotting': {
        'figsize': (12, 8),
        'dpi': 100,
        'colors': {
            'left_contour': 'blue',
            'right_contour': 'red',
            'symmetry_axis': 'green',
            'crossing_lines': 'orange',
            'modules': 'purple'
        },
        'linewidth': 2,
        'markersize': 6
    },

    # Configuración de procesamiento
    'processing': {
        'coordinate_decimals': 4,
        'default_tramo': 2,
        'default_modulo': 5,
        'max_iterations': 1000
    }
}

# Configurar matplotlib
plt.rcParams['figure.figsize'] = CONFIG['plotting']['figsize']
plt.rcParams['figure.dpi'] = CONFIG['plotting']['dpi']

print("✅ Configuración cargada exitosamente")
print(f"📁 Carpeta de datos: {CONFIG['paths']['data_folder']}")
print(f"🎯 Tolerancia geométrica: {CONFIG['tolerances']['geometric']}")

# 02_CARGA ARCHIVOS DXF

In [ ]:
# =====================================================================
# CELDA 3: CARGA DE ARCHIVOS DXF
# =====================================================================

from pathlib import Path
from google.colab import files
import os

def seleccionar_archivo_dxf():
    """
    Busca archivos DXF en /content o permite subirlos mediante diálogo.

    Returns:
        str: Ruta completa al archivo DXF seleccionado

    Raises:
        FileNotFoundError: Si no se encuentra o sube ningún archivo DXF válido
    """
    print("🔍 Buscando archivos DXF...")

    # Buscar archivos DXF existentes en /content
    dxf_files = list(Path("/content").glob("*.dxf"))

    if dxf_files:
        if len(dxf_files) == 1:
            # Si hay solo un archivo, seleccionarlo automáticamente
            selected_file = dxf_files[0]
            print(f"✅ Archivo DXF encontrado: {selected_file.name}")
            print(f"📁 Ruta: {selected_file}")
            return str(selected_file)
        else:
            # Si hay múltiples archivos, mostrar opciones
            print(f"📋 Se encontraron {len(dxf_files)} archivos DXF:")
            for i, file in enumerate(dxf_files, 1):
                print(f"   {i}. {file.name}")

            # Para simplificar, tomar el primero
            selected_file = dxf_files[0]
            print(f"🎯 Seleccionando automáticamente: {selected_file.name}")
            return str(selected_file)

    else:
        # No se encontraron archivos, solicitar carga
        print("❌ No se encontró ningún archivo DXF en /content")
        print("📤 Por favor, sube un archivo DXF:")

        try:
            uploaded = files.upload()

            # Verificar archivos subidos
            for filename in uploaded.keys():
                if filename.lower().endswith(".dxf"):
                    file_path = f"/content/{filename}"
                    print(f"✅ Archivo DXF cargado: {filename}")
                    print(f"📁 Ruta: {file_path}")
                    return file_path

            # Si llegamos aquí, no se subió ningún DXF válido
            raise FileNotFoundError("❌ No se subió ningún archivo DXF válido")

        except Exception as e:
            print(f"❌ Error durante la carga: {str(e)}")
            raise FileNotFoundError("❌ No se pudo cargar el archivo DXF")


def listar_archivos_dxf_disponibles():
    """
    Lista todos los archivos DXF disponibles en /content y subdirectorios.

    Returns:
        list: Lista de rutas de archivos DXF encontrados
    """
    print("🔍 Escaneando archivos DXF disponibles...")

    # Buscar en /content y subdirectorios
    all_dxf_files = []

    # Buscar en directorio principal
    dxf_files_main = list(Path("/content").glob("*.dxf"))
    all_dxf_files.extend(dxf_files_main)

    # Buscar en subdirectorios (hasta 2 niveles)
    dxf_files_sub = list(Path("/content").glob("*/*.dxf"))
    all_dxf_files.extend(dxf_files_sub)

    dxf_files_sub2 = list(Path("/content").glob("*/*/*.dxf"))
    all_dxf_files.extend(dxf_files_sub2)

    if all_dxf_files:
        print(f"📋 Archivos DXF encontrados ({len(all_dxf_files)}):")
        for i, file in enumerate(all_dxf_files, 1):
            file_size = file.stat().st_size / 1024  # KB
            print(f"   {i}. {file.name} ({file_size:.1f} KB)")
            print(f"      📁 {file}")
    else:
        print("❌ No se encontraron archivos DXF")

    return [str(f) for f in all_dxf_files]


def validar_archivo_dxf(file_path: str) -> dict:
    """
    Valida que el archivo DXF sea válido antes de procesarlo.

    Args:
        file_path (str): Ruta al archivo DXF

    Returns:
        dict: Información de validación del archivo
    """
    validation_info = {
        'valid': False,
        'file_exists': False,
        'file_size': 0,
        'is_dxf': False,
        'readable': False,
        'errors': [],
        'warnings': []
    }

    try:
        # Verificar existencia
        if not os.path.exists(file_path):
            validation_info['errors'].append("El archivo no existe")
            return validation_info

        validation_info['file_exists'] = True

        # Verificar tamaño
        file_size = os.path.getsize(file_path)
        validation_info['file_size'] = file_size

        if file_size == 0:
            validation_info['errors'].append("El archivo está vacío")
            return validation_info

        if file_size < 100:  # Menor a 100 bytes es sospechoso
            validation_info['warnings'].append("El archivo es muy pequeño")

        # Verificar extensión
        if not file_path.lower().endswith('.dxf'):
            validation_info['warnings'].append("El archivo no tiene extensión .dxf")
        else:
            validation_info['is_dxf'] = True

        # Verificar que se puede leer
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                first_line = f.readline().strip()
                if 'SECTION' in first_line or 'HEADER' in first_line or first_line == '0':
                    validation_info['readable'] = True
                else:
                    validation_info['warnings'].append("El archivo podría no ser un DXF válido")
        except Exception as e:
            validation_info['errors'].append(f"No se puede leer el archivo: {str(e)}")
            return validation_info

        # Si llegamos aquí, el archivo parece válido
        validation_info['valid'] = True

        # Información adicional
        size_mb = file_size / (1024 * 1024)
        print(f"✅ Archivo DXF válido:")
        print(f"   📁 {os.path.basename(file_path)}")
        print(f"   📏 Tamaño: {size_mb:.2f} MB")

        if validation_info['warnings']:
            print("⚠️ Advertencias:")
            for warning in validation_info['warnings']:
                print(f"   • {warning}")

    except Exception as e:
        validation_info['errors'].append(f"Error inesperado: {str(e)}")

    return validation_info


def cargar_archivo_dxf_con_validacion():
    """
    Función completa para cargar y validar un archivo DXF.

    Returns:
        str: Ruta del archivo DXF válido y listo para procesar

    Raises:
        FileNotFoundError: Si no se puede cargar un archivo válido
    """
    print("🚀 INICIANDO CARGA DE ARCHIVO DXF")
    print("="*40)

    try:
        # Paso 1: Seleccionar archivo
        ruta_dxf = seleccionar_archivo_dxf()

        # Paso 2: Validar archivo
        print("\n🔍 Validando archivo...")
        validation = validar_archivo_dxf(ruta_dxf)

        if not validation['valid']:
            print("❌ El archivo no es válido:")
            for error in validation['errors']:
                print(f"   • {error}")
            raise FileNotFoundError("Archivo DXF no válido")

        if validation['warnings']:
            print("⚠️ Advertencias (el archivo se puede usar):")
            for warning in validation['warnings']:
                print(f"   • {warning}")

        print(f"\n✅ Archivo DXF listo para procesar: {os.path.basename(ruta_dxf)}")
        return ruta_dxf

    except Exception as e:
        print(f"❌ Error en la carga: {str(e)}")
        raise


# Ejecutar la carga automática del archivo DXF
print("🎯 CARGANDO ARCHIVO DXF AUTOMÁTICAMENTE...")
try:
    ruta_dxf = cargar_archivo_dxf_con_validacion()
    print(f"\n🎉 ¡Archivo listo! Variable 'ruta_dxf' configurada con: {ruta_dxf}")

    # También crear variable global para fácil acceso
    DXF_FILE_PATH = ruta_dxf

except Exception as e:
    print(f"\n⚠️ No se pudo cargar automáticamente. Error: {str(e)}")
    print("💡 Puedes intentar manualmente con:")
    print("   ruta_dxf = seleccionar_archivo_dxf()")
    ruta_dxf = None

print("\n" + "="*50)
print("✅ Módulo de carga de archivos DXF listo")
if 'ruta_dxf' in locals() and ruta_dxf:
    print(f"✅ Archivo DXF configurado: {os.path.basename(ruta_dxf)}")
else:
    print("⚠️ No hay archivo DXF configurado aún")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 03_EXTRAE LÍNEAS DEL DXF + CONTORNOS

# Nota: El resto del código ha sido truncado por límites de longitud.
# El notebook completo contiene análisis de torres, módulos y generación de parámetros.